# PCA vs MS-AA network-motif simulation benchmark

This notebook is the **clean primary simulation analysis** for the manuscript revision.

The main scientific comparison is now deliberately limited to **PCA versus
multisubject archetypal analysis (MS-AA)**:

1. **PCA-favorable control** — verifies that the benchmark can recover
   variance-defined orthogonal directions when PCA assumptions are correct.
2. **AA-favorable control** — verifies that the benchmark can recover
   extremal convex-mixture motifs when AA assumptions are correct.
3. **Brain-inspired metastable regime** — the key scientific test. Recurring,
   overlapping distributed configurations have persistence, gradual
   transitions, coactivation, subject variability, noise, time-varying gain,
   and signed autocorrelated deviations. The generator is therefore **not an
   exact PCA or AA model**.

The primary question is not whether MS-AA reconstructs the data better than
PCA. It is whether MS-AA more directly identifies recurring distributed
configurations while PCA may retain the same information in a rotated
variance-based basis.

The notebook also retains the planned sensitivity analyses for noise,
between-subject heterogeneity, fitted model order, motif overlap, dynamical
mixing, temporal activation recovery, and normalization equivalence.

### ICA

ICA is **not part of the primary analysis**. A single optional supplementary
calibration cell is provided at the end and is off by default. Its sole
purpose is to verify that an ICA-favorable independent-source simulation is
indeed recovered best by ICA; those results do not enter the primary
checkpoint, figures, or inference.


In [ ]:
import numpy as np

# ============================================================
# Configuration
# ============================================================

RUN_MODE = "paper"   # "quick" or "paper"
RNG_SEED = 42

HELPERS_DIR = "helpers"
HYPERTOOLS_NORMALIZE_PATH = "hypertools/normalize.py"

# New checkpoint namespace: main analysis is now PCA vs MS-AA only.
OUTPUT_DIR = "simulation_pca_msaa_extended_outputs"

ANALYSIS_TYPE = "spatial"

MSAA_OPTS_BASE = dict(
    maxiter=200,
    conv_crit=1e-6,
    fix_var_iter=5,
    heteroscedastic=False,
    use_gpu=False,
    rngSEED=RNG_SEED,
)

# Shared spatial architecture
N_NETWORKS = 6
SUBNETWORKS_PER_NETWORK = 2
NODES_PER_SUBNETWORK = 10
N_TRUE_MOTIFS = 6

BASE_NOISE_SD = 0.25
BASE_MULTIPLICATIVE_SD = 0.08
BASE_ADDITIVE_SD = 0.04

# PCA-positive-control source standard deviations.
PCA_SOURCE_SCALES = np.array(
    [2.0, 1.7, 1.4, 1.1, 0.8, 0.5]
)

# AA-positive-control geometry
DIRICHLET_ALPHA = 0.10

# Brain-inspired metastable dynamics
MEAN_DWELL = 12
TRANSITION_WIDTH = 4
SECONDARY_WEIGHT = 0.25
AR1_RHO = 0.85

# Make metastable dynamics deliberately non-simplex-exact.
GLOBAL_GAIN_SD = 0.15
SIGNED_DEVIATION_SD = 0.08
DEVIATION_RHO = 0.80

# Motif-overlap manipulation.
# Higher distinctiveness -> lower overlap.
OVERLAP_LEVELS = {
    "low_overlap": 1.15,
    "baseline_overlap": 1.00,
    "high_overlap": 0.55,
}

# Joint dynamics manipulation.
DYNAMICS_PRESETS = {
    "state_like": {
        "mean_dwell": 16,
        "transition_width": 2,
        "secondary_weight": 0.10,
        "ar1_rho": 0.75,
    },
    "baseline": {
        "mean_dwell": 12,
        "transition_width": 4,
        "secondary_weight": 0.25,
        "ar1_rho": 0.85,
    },
    "more_mixed": {
        "mean_dwell": 8,
        "transition_width": 8,
        "secondary_weight": 0.45,
        "ar1_rho": 0.90,
    },
}

# Secondary robustness
NOISE_LEVELS = [0.10, 0.25, 0.40]
HETEROGENEITY_LEVELS = [0.0, 1.0, 1.5]
K_LEVELS = [4, 6, 8]

# Optional supplementary ICA calibration.
# This is OFF by default and never enters the main checkpoint/results.
RUN_OPTIONAL_ICA_CALIBRATION = False

if RUN_MODE == "quick":
    N_SUBJECTS = 10
    T_PER_SUBJECT = 120
    N_REPS_CORE = 2
    N_REPS_METASTABLE = 3
    N_REPS_ROBUST = 1
    N_REPS_SENSITIVITY = 1
    OPTIONAL_ICA_REPS = 2
elif RUN_MODE == "paper":
    N_SUBJECTS = 16
    T_PER_SUBJECT = 180
    N_REPS_CORE = 20
    N_REPS_METASTABLE = 20
    N_REPS_ROBUST = 10
    N_REPS_SENSITIVITY = 10
    OPTIONAL_ICA_REPS = 5
else:
    raise ValueError(
        "RUN_MODE must be 'quick' or 'paper'"
    )

print("RUN_MODE:", RUN_MODE)
print("Subjects:", N_SUBJECTS)
print("Time points per subject:", T_PER_SUBJECT)
print("Core reps:", N_REPS_CORE)
print("Metastable reps:", N_REPS_METASTABLE)
print("Robustness reps:", N_REPS_ROBUST)
print("Sensitivity reps:", N_REPS_SENSITIVITY)
print("Optional ICA calibration:", RUN_OPTIONAL_ICA_CALIBRATION)


In [ ]:
# ============================================================
# Imports
# ============================================================

%matplotlib inline

import os
import sys
import time
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import PCA

sys.path.append(HELPERS_DIR)
from MultiSubject_AA import Subject, multi_subject_aa

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RESULTS_CSV = OUTPUT_DIR / (
    f"pca_msaa_extended_results_{RUN_MODE}.csv"
)

print("Checkpoint:", RESULTS_CSV)


## 1. MS-AA helpers matching the empirical pipeline


In [ ]:
def coerce_TxV(x):
    x = np.asarray(x, dtype=float)

    if x.ndim != 2:
        raise ValueError(f"Expected 2D T x V matrix, got {x.shape}")

    x = x - x.mean(axis=0, keepdims=True)
    return x.astype(np.float32, copy=False)


def orient_subject_matrix(X_raw, analysis_type):
    Xtv = coerce_TxV(X_raw)
    return Xtv if analysis_type == "spatial" else Xtv.T


def build_subject_objects(subject_list, analysis_type):
    return [
        Subject(
            X=orient_subject_matrix(x, analysis_type),
            sX=orient_subject_matrix(x, analysis_type),
        )
        for x in subject_list
    ]


def fit_msaa(subject_list, K, analysis_type="spatial", seed=RNG_SEED, verbose=False):
    opts = dict(MSAA_OPTS_BASE)
    opts["rngSEED"] = int(seed)

    subj_objs = build_subject_objects(subject_list, analysis_type)

    t0 = time.time()

    results_subj, C, cost_fun, varexpl, secs = multi_subject_aa(
        subj_objs,
        noc=int(K),
        opts=opts,
    )

    elapsed = time.time() - t0

    if verbose:
        print(
            f"[MS-AA] K={K} | VarExpl={varexpl*100:.2f}% | "
            f"{elapsed:.2f}s"
        )

    return {
        "results_subj": results_subj,
        "C": C,
        "cost_fun": cost_fun,
        "varexpl": varexpl,
        "secs": secs,
        "elapsed": elapsed,
        "K": int(K),
    }


def reconstruct_from_archetypes(sub, analysis_type="spatial"):
    sXC = np.asarray(sub["sXC"], dtype=float)
    S = np.asarray(sub["S"], dtype=float)

    if analysis_type == "spatial":
        return sXC @ S

    return (sXC @ S).T


## 2. Distributed network-motif geometry


In [ ]:
def make_overlapping_motifs():
    n_subnetworks = (
        N_NETWORKS
        * SUBNETWORKS_PER_NETWORK
    )

    V = (
        n_subnetworks
        * NODES_PER_SUBNETWORK
    )

    motifs = np.full(
        (N_TRUE_MOTIFS, V),
        0.05,
        dtype=float,
    )

    motif_subnetworks = [
        [0, 3, 8],
        [1, 3, 9],
        [2, 5, 10],
        [2, 4, 11],
        [0, 6, 7],
        [1, 6, 8],
    ]

    for k, active in enumerate(
        motif_subnetworks
    ):
        for subnet in active:
            lo = (
                subnet
                * NODES_PER_SUBNETWORK
            )
            hi = (
                lo
                + NODES_PER_SUBNETWORK
            )

            motifs[
                k,
                lo:hi,
            ] = 1.0

    return motifs


def make_overlap_variant(
    base_motifs,
    distinctiveness_scale,
):
    # Scale each motif's deviation from the common motif.
    # This preserves network topology while changing
    # nonorthogonality/overlap.
    base_motifs = np.asarray(
        base_motifs,
        dtype=float,
    )

    common = base_motifs.mean(
        axis=0,
        keepdims=True,
    )

    return (
        common
        + distinctiveness_scale
        * (
            base_motifs
            - common
        )
    )


def mean_pairwise_cosine(patterns):
    patterns = np.asarray(
        patterns,
        dtype=float,
    )

    unit = patterns / (
        np.linalg.norm(
            patterns,
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    C = unit @ unit.T

    tri = np.triu_indices(
        C.shape[0],
        k=1,
    )

    return float(
        np.mean(C[tri])
    )


TRUE_MOTIFS_RAW = (
    make_overlapping_motifs()
)

V = TRUE_MOTIFS_RAW.shape[1]

print(
    "Motif matrix:",
    TRUE_MOTIFS_RAW.shape,
)

for name, scale in (
    OVERLAP_LEVELS.items()
):
    variant = make_overlap_variant(
        TRUE_MOTIFS_RAW,
        scale,
    )

    print(
        name,
        "| distinctiveness =",
        scale,
        "| mean pairwise cosine =",
        round(
            mean_pairwise_cosine(
                variant
            ),
            3,
        ),
    )


In [ ]:
def plot_pattern_matrix(patterns, title, ylabel="Pattern"):
    fig, ax = plt.subplots(figsize=(12, 4))
    im = ax.imshow(patterns, aspect="auto")

    for x in range(
        NODES_PER_SUBNETWORK,
        patterns.shape[1],
        NODES_PER_SUBNETWORK,
    ):
        ax.axvline(x - 0.5, linewidth=0.4, alpha=0.35)

    network_width = (
        SUBNETWORKS_PER_NETWORK
        * NODES_PER_SUBNETWORK
    )

    for x in range(
        network_width,
        patterns.shape[1],
        network_width,
    ):
        ax.axvline(x - 0.5, linewidth=1.2)

    ax.set_xlabel("Nodes grouped by network → subnetwork")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8)

    plt.show()


plot_pattern_matrix(
    TRUE_MOTIFS_RAW,
    "Ground-truth overlapping distributed network motifs",
    ylabel="True motif",
)


## 3. Normalization and recovery metrics

The simulation uses the same subject-wise centering followed by across-subject
feature scaling as the empirical pipeline.

Ground truth is transformed differently depending on whether it represents a
**direction** (PCA control) or a **point/state template** (AA and metastable
regimes). This avoids incorrectly translating variance axes during centering.


In [ ]:
def normalize_across_subjects(
    subject_list,
    eps=1e-8,
):
    # Match the empirical fallback:
    # coerce/center EACH subject first,
    # then scale features across the stacked subjects.
    raw = [
        np.asarray(
            x,
            dtype=float,
        )
        for x in subject_list
    ]

    subject_means = [
        x.mean(
            axis=0,
            keepdims=True,
        )
        for x in raw
    ]

    centered = [
        x - mu_s
        for x, mu_s in zip(
            raw,
            subject_means,
        )
    ]

    stacked = np.vstack(
        centered
    )

    across_mu = stacked.mean(
        axis=0,
        keepdims=True,
    )

    sd = (
        stacked.std(
            axis=0,
            keepdims=True,
        )
        + eps
    )

    normalized = [
        (
            (x - across_mu)
            / sd
        ).astype(
            np.float32,
            copy=False,
        )
        for x in centered
    ]

    mean_subject_mean = np.mean(
        np.stack(
            subject_means,
            axis=0,
        ),
        axis=0,
    )

    return (
        normalized,
        mean_subject_mean,
        sd,
    )


def transform_true_patterns(
    true_patterns_raw,
    mean_subject_mean,
    sd,
    truth_kind,
):
    true_patterns_raw = np.asarray(
        true_patterns_raw,
        dtype=float,
    )

    if truth_kind == "direction":
        # Centering observations changes origins,
        # not generative directions.
        return (
            true_patterns_raw
            / sd
        )

    if truth_kind == "point":
        # State/archetype templates are points.
        # Averaging subject-wise affine transforms
        # gives this group-level transform.
        return (
            (
                true_patterns_raw
                - mean_subject_mean
            )
            / sd
        )

    raise ValueError(
        "truth_kind must be "
        "'direction' or 'point'"
    )


def pattern_corr_matrix(
    A,
    B,
    absolute=False,
):
    A = np.asarray(
        A,
        dtype=float,
    )

    B = np.asarray(
        B,
        dtype=float,
    )

    A0 = (
        A
        - A.mean(
            axis=1,
            keepdims=True,
        )
    )

    B0 = (
        B
        - B.mean(
            axis=1,
            keepdims=True,
        )
    )

    A0 /= (
        np.linalg.norm(
            A0,
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    B0 /= (
        np.linalg.norm(
            B0,
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    C = A0 @ B0.T

    return (
        np.abs(C)
        if absolute
        else C
    )


def optimal_match(
    true_patterns,
    recovered_patterns,
    absolute=False,
):
    C = pattern_corr_matrix(
        true_patterns,
        recovered_patterns,
        absolute=absolute,
    )

    (
        true_idx,
        recovered_idx,
    ) = linear_sum_assignment(
        -C
    )

    return {
        "similarity": C,
        "true_idx": true_idx,
        "recovered_idx": recovered_idx,
        "matched": C[
            true_idx,
            recovered_idx,
        ],
    }


def coverage_adjusted_recovery(
    match,
    n_true,
):
    return float(
        np.sum(
            match["matched"]
        )
        / n_true
    )



def temporal_recovery_from_spatial_match(
    true_activations,
    recovered_scores,
    spatial_match,
    absolute=False,
):
    # Temporal recovery is anchored to spatial matching.
    # Because simulated observations are centered within subject,
    # center latent/recovered trajectories within subject before
    # concatenating across subjects.
    if true_activations is None:
        return {
            "mean": np.nan,
            "coverage": np.nan,
            "per_pattern": (
                np.array([])
            ),
        }

    true_list = [
        np.asarray(
            a,
            dtype=float,
        )
        for a in true_activations
    ]

    true_centered = [
        a - a.mean(
            axis=0,
            keepdims=True,
        )
        for a in true_list
    ]

    true_stack = np.vstack(
        true_centered
    )

    if isinstance(
        recovered_scores,
        (list, tuple),
    ):
        recovered_list = [
            np.asarray(
                a,
                dtype=float,
            )
            for a in recovered_scores
        ]
    else:
        recovered_array = np.asarray(
            recovered_scores,
            dtype=float,
        )

        lengths = [
            a.shape[0]
            for a in true_list
        ]

        cuts = np.cumsum(
            lengths
        )[:-1]

        recovered_list = (
            np.split(
                recovered_array,
                cuts,
                axis=0,
            )
        )

    recovered_centered = [
        a - a.mean(
            axis=0,
            keepdims=True,
        )
        for a in recovered_list
    ]

    recovered_stack = np.vstack(
        recovered_centered
    )

    vals = []

    for (
        true_idx,
        recovered_idx,
    ) in zip(
        spatial_match["true_idx"],
        spatial_match[
            "recovered_idx"
        ],
    ):
        a = true_stack[
            :,
            true_idx,
        ]

        b = recovered_stack[
            :,
            recovered_idx,
        ]

        if (
            np.std(a) < 1e-12
            or np.std(b) < 1e-12
        ):
            r = np.nan
        else:
            r = np.corrcoef(
                a,
                b,
            )[0, 1]

        if (
            absolute
            and np.isfinite(r)
        ):
            r = abs(r)

        vals.append(r)

    vals = np.asarray(
        vals,
        dtype=float,
    )

    return {
        "mean": float(
            np.nanmean(vals)
        ),
        "coverage": float(
            np.nansum(vals)
            / true_stack.shape[1]
        ),
        "per_pattern": vals,
    }

def collapse_to_subnetworks(
    patterns,
):
    patterns = np.asarray(
        patterns,
        dtype=float,
    )

    n_subnetworks = (
        N_NETWORKS
        * SUBNETWORKS_PER_NETWORK
    )

    out = np.zeros(
        (
            patterns.shape[0],
            n_subnetworks,
        )
    )

    for subnet in range(
        n_subnetworks
    ):
        lo = (
            subnet
            * NODES_PER_SUBNETWORK
        )

        hi = (
            lo
            + NODES_PER_SUBNETWORK
        )

        out[
            :,
            subnet,
        ] = (
            patterns[
                :,
                lo:hi,
            ].mean(
                axis=1
            )
        )

    return out


def linear_span_recovery(
    true_patterns,
    recovered_patterns,
):
    # Project true patterns onto the row span
    # of the recovered patterns.
    B = np.asarray(
        recovered_patterns,
        dtype=float,
    ).T

    vals = []

    for true in np.asarray(
        true_patterns,
        dtype=float,
    ):
        coef, *_ = np.linalg.lstsq(
            B,
            true,
            rcond=None,
        )

        recon = (
            B @ coef
        )

        vals.append(
            np.corrcoef(
                true,
                recon,
            )[0, 1]
        )

    return float(
        np.nanmean(vals)
    )


## 4. Matched PCA vs MS-AA evaluator

Both methods are fit at the same \(K\). We evaluate:

- **spatial motif recovery**: one-to-one correspondence with the true spatial
  configurations;
- **temporal recovery**: whether the same spatially matched dimension recovers
  when that configuration was expressed;
- **subnetwork recovery**: whether the network composition is preserved;
- **span recovery**: whether the true configuration remains encoded somewhere
  in the recovered low-dimensional subspace;
- **reconstruction MSE**: descriptive fidelity to the observed data.

For fitted \(K<K_{\mathrm{true}}\), coverage-adjusted recovery prevents a
method from appearing strong merely because it recovered a subset of motifs.


In [ ]:
def evaluate_methods(
    subject_data,
    true_patterns_raw,
    K_fit,
    seed,
    regime,
    rep,
    true_activations=None,
    truth_kind="point",
    extra=None,
    verbose=False,
):
    """
    Matched PCA vs MS-AA evaluation.

    Primary outcomes:
      - direct spatial pattern recovery
      - temporal activation recovery anchored to the spatial match
      - subnetwork-composition recovery
      - latent-span recovery
      - reconstruction MSE

    PCA spatial/temporal correlations are sign-invariant because PCA axes have
    arbitrary sign. MS-AA uses signed correlations.
    """
    (
        subjects,
        mean_subject_mean,
        sd,
    ) = normalize_across_subjects(
        subject_data
    )

    true_patterns = transform_true_patterns(
        true_patterns_raw,
        mean_subject_mean,
        sd,
        truth_kind=truth_kind,
    )

    X_stack = np.vstack(
        subjects
    )

    # ========================================================
    # MS-AA
    # ========================================================

    msaa = fit_msaa(
        subjects,
        K=K_fit,
        analysis_type="spatial",
        seed=seed,
        verbose=verbose,
    )

    S_group = np.mean(
        np.stack(
            [
                np.asarray(
                    sub["S"],
                    dtype=float,
                )
                for sub in msaa[
                    "results_subj"
                ]
            ],
            axis=0,
        ),
        axis=0,
    )

    aa_scores = [
        np.asarray(
            sub["sXC"],
            dtype=float,
        )
        for sub in msaa[
            "results_subj"
        ]
    ]

    Xhat_aa = np.vstack([
        reconstruct_from_archetypes(
            sub,
            "spatial",
        )
        for sub in msaa[
            "results_subj"
        ]
    ])

    aa_mse = float(
        np.mean(
            (X_stack - Xhat_aa) ** 2
        )
    )

    # ========================================================
    # PCA
    # ========================================================

    pca = PCA(
        n_components=int(K_fit),
        random_state=int(seed),
    )

    pca_scores = pca.fit_transform(
        X_stack
    )

    Xhat_pca = pca.inverse_transform(
        pca_scores
    )

    pca_patterns = np.asarray(
        pca.components_,
        dtype=float,
    )

    pca_mse = float(
        np.mean(
            (X_stack - Xhat_pca) ** 2
        )
    )

    # ========================================================
    # Spatial recovery
    # ========================================================

    aa_match = optimal_match(
        true_patterns,
        S_group,
        absolute=False,
    )

    pca_match = optimal_match(
        true_patterns,
        pca_patterns,
        absolute=True,
    )

    # ========================================================
    # Temporal recovery anchored to the spatial match
    # ========================================================

    aa_temporal = temporal_recovery_from_spatial_match(
        true_activations,
        aa_scores,
        aa_match,
        absolute=False,
    )

    pca_temporal = temporal_recovery_from_spatial_match(
        true_activations,
        pca_scores,
        pca_match,
        absolute=True,
    )

    # ========================================================
    # Subnetwork recovery
    # ========================================================

    true_sub = collapse_to_subnetworks(
        true_patterns
    )

    aa_sub = collapse_to_subnetworks(
        S_group
    )

    pca_sub = collapse_to_subnetworks(
        pca_patterns
    )

    aa_sub_match = optimal_match(
        true_sub,
        aa_sub,
        absolute=False,
    )

    pca_sub_match = optimal_match(
        true_sub,
        pca_sub,
        absolute=True,
    )

    # ========================================================
    # Span recovery
    # ========================================================

    aa_span = linear_span_recovery(
        true_patterns,
        S_group,
    )

    pca_span = linear_span_recovery(
        true_patterns,
        pca_patterns,
    )

    n_true = true_patterns.shape[0]

    row = {
        "regime": regime,
        "condition": "base",
        "rep": int(rep),
        "seed": int(seed),
        "K_fit": int(K_fit),
        "truth_kind": truth_kind,

        "aa_motif_recovery": float(
            np.mean(
                aa_match["matched"]
            )
        ),
        "pca_motif_recovery": float(
            np.mean(
                pca_match["matched"]
            )
        ),

        "aa_motif_recovery_coverage": (
            coverage_adjusted_recovery(
                aa_match,
                n_true,
            )
        ),
        "pca_motif_recovery_coverage": (
            coverage_adjusted_recovery(
                pca_match,
                n_true,
            )
        ),

        "aa_temporal_recovery": (
            aa_temporal["mean"]
        ),
        "pca_temporal_recovery": (
            pca_temporal["mean"]
        ),

        "aa_temporal_recovery_coverage": (
            aa_temporal["coverage"]
        ),
        "pca_temporal_recovery_coverage": (
            pca_temporal["coverage"]
        ),

        "aa_subnetwork_recovery": float(
            np.mean(
                aa_sub_match["matched"]
            )
        ),
        "pca_subnetwork_recovery": float(
            np.mean(
                pca_sub_match["matched"]
            )
        ),

        "aa_span_recovery": aa_span,
        "pca_span_recovery": pca_span,

        "aa_mse": aa_mse,
        "pca_mse": pca_mse,

        "aa_varexpl": float(
            msaa["varexpl"]
        ),
        "pca_explained_variance": float(
            np.sum(
                pca.explained_variance_ratio_
            )
        ),

        "msaa_elapsed_sec": float(
            msaa["elapsed"]
        ),
    }

    if extra:
        row.update(extra)

    return row


# REGIME 1 — PCA-favorable control

The true spatial factors are orthogonal covariance directions with distinct
source variances. PCA should recover those generating directions most
directly. MS-AA is being asked to represent a geometry that is not
archetypal.


In [ ]:
def simulate_pca_control(seed):
    rng = np.random.default_rng(
        seed
    )

    Q, _ = np.linalg.qr(
        rng.normal(
            size=(
                V,
                N_TRUE_MOTIFS,
            )
        )
    )

    true_factors = Q.T

    subjects = []
    activations = []

    for s in range(
        N_SUBJECTS
    ):
        scores = rng.normal(
            size=(
                T_PER_SUBJECT,
                N_TRUE_MOTIFS,
            )
        )

        scores = (
            scores
            * PCA_SOURCE_SCALES[
                None,
                :,
            ]
        )

        X = (
            scores
            @ true_factors
        )

        X += rng.normal(
            0,
            BASE_NOISE_SD,
            size=X.shape,
        )

        subjects.append(X)
        activations.append(
            scores
        )

    return {
        "subjects": subjects,
        "true_patterns": (
            true_factors
        ),
        "activations": (
            activations
        ),
        "truth_kind": "direction",
    }


# REGIME 2 — AA-favorable control

Each observation is a sparse convex mixture of overlapping distributed
extremal motifs, with mild subject-specific perturbation. MS-AA should recover
the generating motifs most directly. PCA may still preserve the underlying
low-dimensional span.


In [ ]:
def simulate_aa_control(
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    subjects = []
    activations = []

    for s in range(
        N_SUBJECTS
    ):
        multiplicative = (
            1.0
            + rng.normal(
                0,
                BASE_MULTIPLICATIVE_SD,
                size=(
                    TRUE_MOTIFS_RAW.shape
                ),
            )
        )

        additive = rng.normal(
            0,
            BASE_ADDITIVE_SD,
            size=(
                TRUE_MOTIFS_RAW.shape
            ),
        )

        subj_motifs = (
            TRUE_MOTIFS_RAW
            * multiplicative
            + additive
        )

        W = rng.dirichlet(
            np.full(
                N_TRUE_MOTIFS,
                DIRICHLET_ALPHA,
            ),
            size=T_PER_SUBJECT,
        )

        X = (
            W @ subj_motifs
        )

        X += rng.normal(
            0,
            BASE_NOISE_SD,
            size=X.shape,
        )

        subjects.append(X)
        activations.append(W)

    return {
        "subjects": subjects,
        "true_patterns": (
            TRUE_MOTIFS_RAW
        ),
        "activations": (
            activations
        ),
        "truth_kind": "point",
    }


# REGIME 3 — Brain-inspired metastable dynamics

This is the key scientific comparison.

Distributed configurations recur over time, persist for variable dwell
periods, transition gradually, partially coactivate, and vary across subjects.
After this state-like process is generated, the latent activations receive
time-varying global gain and signed autocorrelated deviations:

\[
X_t = \sum_k A^\ast_{tk}M_k^{(s)} + \epsilon_t.
\]

Unlike the AA-positive control, \(A^\ast\) is **not constrained to be
nonnegative or to sum to one**. Thus the metastable regime is not a literal
simplex generator.


In [ ]:
def sample_state_sequence(
    T,
    K,
    rng,
    mean_dwell=MEAN_DWELL,
):
    states = []
    current = int(
        rng.integers(K)
    )

    while len(states) < T:
        dwell = max(
            2,
            int(
                rng.geometric(
                    1.0
                    / mean_dwell
                )
            ),
        )

        states.extend(
            [current] * dwell
        )

        candidates = [
            k
            for k in range(K)
            if k != current
        ]

        current = int(
            rng.choice(
                candidates
            )
        )

    return np.asarray(
        states[:T],
        dtype=int,
    )


def smooth_state_activations(
    states,
    K,
    transition_width=(
        TRANSITION_WIDTH
    ),
):
    T = len(states)

    A = np.zeros(
        (T, K),
        dtype=float,
    )

    A[
        np.arange(T),
        states,
    ] = 1.0

    if transition_width > 0:
        kernel_x = np.arange(
            -transition_width,
            transition_width + 1,
        )

        sigma = max(
            1.0,
            transition_width / 2,
        )

        kernel = np.exp(
            -(
                kernel_x ** 2
            )
            / (
                2
                * sigma ** 2
            )
        )

        kernel /= kernel.sum()

        for k in range(K):
            A[:, k] = np.convolve(
                A[:, k],
                kernel,
                mode="same",
            )

    A /= (
        A.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return A


def ar1_process(
    T,
    K,
    rho,
    rng,
):
    x = np.zeros(
        (T, K),
        dtype=float,
    )

    eps = rng.normal(
        0,
        1,
        size=(T, K),
    )

    x[0] = eps[0]

    innovation_scale = np.sqrt(
        max(
            1e-12,
            1.0 - rho ** 2,
        )
    )

    for t in range(
        1,
        T,
    ):
        x[t] = (
            rho * x[t - 1]
            + innovation_scale
            * eps[t]
        )

    return x


def add_correlated_secondary_recruitment(
    A,
    rng,
    secondary_weight=(
        SECONDARY_WEIGHT
    ),
    ar1_rho=AR1_RHO,
):
    T, K = A.shape

    background = ar1_process(
        T,
        K,
        ar1_rho,
        rng,
    )

    background = np.exp(
        background
    )

    background /= (
        background.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    mixed = (
        (
            1.0
            - secondary_weight
        )
        * A
        + secondary_weight
        * background
    )

    mixed /= (
        mixed.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return mixed


def add_non_simplex_modulation(
    A,
    rng,
    global_gain_sd=(
        GLOBAL_GAIN_SD
    ),
    signed_deviation_sd=(
        SIGNED_DEVIATION_SD
    ),
    deviation_rho=(
        DEVIATION_RHO
    ),
):
    T, K = A.shape

    gain_latent = ar1_process(
        T,
        1,
        deviation_rho,
        rng,
    )[:, 0]

    gain = np.exp(
        global_gain_sd
        * gain_latent
        - 0.5
        * global_gain_sd ** 2
    )

    signed_dev = ar1_process(
        T,
        K,
        deviation_rho,
        rng,
    )

    A_eff = (
        gain[:, None] * A
        + signed_deviation_sd
        * signed_dev
    )

    # Deliberately no nonnegative projection
    # and no row-sum normalization here.
    return A_eff, gain


def simulate_metastable(
    seed,
    noise_sd=BASE_NOISE_SD,
    heterogeneity_scale=1.0,
    motifs_raw=None,
    mean_dwell=MEAN_DWELL,
    transition_width=(
        TRANSITION_WIDTH
    ),
    secondary_weight=(
        SECONDARY_WEIGHT
    ),
    ar1_rho=AR1_RHO,
):
    rng = np.random.default_rng(
        seed
    )

    if motifs_raw is None:
        motifs_raw = (
            TRUE_MOTIFS_RAW
        )

    motifs_raw = np.asarray(
        motifs_raw,
        dtype=float,
    )

    subjects = []
    state_sequences = []
    activations = []
    simplex_activations = []
    gains = []

    for s in range(
        N_SUBJECTS
    ):
        multiplicative = (
            1.0
            + rng.normal(
                0,
                BASE_MULTIPLICATIVE_SD
                * heterogeneity_scale,
                size=motifs_raw.shape,
            )
        )

        additive = rng.normal(
            0,
            BASE_ADDITIVE_SD
            * heterogeneity_scale,
            size=motifs_raw.shape,
        )

        subj_motifs = (
            motifs_raw
            * multiplicative
            + additive
        )

        states = (
            sample_state_sequence(
                T_PER_SUBJECT,
                N_TRUE_MOTIFS,
                rng,
                mean_dwell=(
                    mean_dwell
                ),
            )
        )

        A_state = (
            smooth_state_activations(
                states,
                N_TRUE_MOTIFS,
                transition_width=(
                    transition_width
                ),
            )
        )

        A_simplex = (
            add_correlated_secondary_recruitment(
                A_state,
                rng,
                secondary_weight=(
                    secondary_weight
                ),
                ar1_rho=(
                    ar1_rho
                ),
            )
        )

        (
            A_eff,
            gain,
        ) = (
            add_non_simplex_modulation(
                A_simplex,
                rng,
            )
        )

        X = (
            A_eff @ subj_motifs
        )

        X += rng.normal(
            0,
            noise_sd,
            size=X.shape,
        )

        subjects.append(X)
        state_sequences.append(
            states
        )
        activations.append(
            A_eff
        )
        simplex_activations.append(
            A_simplex
        )
        gains.append(gain)

    return {
        "subjects": subjects,
        "true_patterns": motifs_raw,
        "states": state_sequences,
        "activations": activations,
        "simplex_activations": (
            simplex_activations
        ),
        "gains": gains,
        "truth_kind": "point",
        "dynamics": {
            "mean_dwell": (
                mean_dwell
            ),
            "transition_width": (
                transition_width
            ),
            "secondary_weight": (
                secondary_weight
            ),
            "ar1_rho": (
                ar1_rho
            ),
        },
    }


## 5. Metastable-process diagnostics


In [ ]:
def metastable_diagnostics(
    sim,
):
    dwell_lengths = []

    for states in sim["states"]:
        start = 0

        for t in range(
            1,
            len(states) + 1,
        ):
            if (
                t == len(states)
                or states[t]
                != states[start]
            ):
                dwell_lengths.append(
                    t - start
                )

                start = t

    lag1 = []

    for A in sim[
        "activations"
    ]:
        for k in range(
            A.shape[1]
        ):
            lag1.append(
                np.corrcoef(
                    A[:-1, k],
                    A[1:, k],
                )[0, 1]
            )

    all_A = np.vstack(
        sim["activations"]
    )

    row_sums = all_A.sum(
        axis=1
    )

    return {
        "mean_dwell": float(
            np.mean(
                dwell_lengths
            )
        ),
        "median_dwell": float(
            np.median(
                dwell_lengths
            )
        ),
        "mean_activation_lag1": float(
            np.nanmean(
                lag1
            )
        ),
        "activation_row_sum_sd": float(
            np.std(
                row_sums
            )
        ),
        "fraction_negative_coefficients": float(
            np.mean(
                all_A < 0
            )
        ),
    }


demo = simulate_metastable(
    RNG_SEED
)

print(
    metastable_diagnostics(
        demo
    )
)

fig, ax = plt.subplots(
    figsize=(10, 4)
)

ax.plot(
    demo["activations"][0]
)

ax.set_xlabel("Time")
ax.set_ylabel(
    "Motif activation"
)

ax.set_title(
    "Example non-simplex metastable activation trajectories"
)

plt.show()


## 6. Restart-safe checkpoints


In [ ]:
def load_results():
    if RESULTS_CSV.exists():
        df = pd.read_csv(RESULTS_CSV)
        print(
            f"Loaded {len(df)} completed simulations"
        )
        return df

    return pd.DataFrame()


def is_complete(
    df,
    regime,
    condition,
    rep,
):
    if df.empty:
        return False

    mask = (
        (df["regime"] == regime)
        & (df["condition"] == condition)
        & (
            df["rep"].astype(int)
            == int(rep)
        )
    )

    return bool(mask.any())


def save_row(df, row):
    new = pd.DataFrame([row])

    out = (
        new
        if df.empty
        else pd.concat(
            [df, new],
            ignore_index=True,
        )
    )

    out = out.drop_duplicates(
        subset=[
            "regime",
            "condition",
            "rep",
        ],
        keep="last",
    )

    tmp = RESULTS_CSV.with_suffix(
        ".tmp.csv"
    )

    out.to_csv(
        tmp,
        index=False,
    )

    os.replace(
        tmp,
        RESULTS_CSV,
    )

    return out


# PART A — Core three-regime benchmark


In [ ]:
results_df = load_results()

core_specs = [
    (
        "PCA_control",
        simulate_pca_control,
        N_REPS_CORE,
        101,
    ),
    (
        "AA_control",
        simulate_aa_control,
        N_REPS_CORE,
        202,
    ),
]

for (
    regime,
    generator,
    n_reps,
    regime_code,
) in core_specs:

    for rep in range(
        n_reps
    ):
        condition = "base"

        if is_complete(
            results_df,
            regime,
            condition,
            rep,
        ):
            print(
                f"SKIP {regime} rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [
                    RNG_SEED,
                    regime_code,
                    rep,
                ]
            ).generate_state(1)[0]
        )

        sim = generator(seed)

        row = evaluate_methods(
            sim["subjects"],
            sim["true_patterns"],
            K_fit=N_TRUE_MOTIFS,
            seed=seed,
            regime=regime,
            rep=rep,
            true_activations=sim[
                "activations"
            ],
            truth_kind=sim[
                "truth_kind"
            ],
            extra={
                "condition": condition
            },
        )

        results_df = save_row(
            results_df,
            row,
        )


for rep in range(
    N_REPS_METASTABLE
):
    regime = "metastable"
    condition = "base"

    if is_complete(
        results_df,
        regime,
        condition,
        rep,
    ):
        print(
            f"SKIP {regime} rep={rep}"
        )
        continue

    seed = int(
        np.random.SeedSequence(
            [
                RNG_SEED,
                303,
                rep,
            ]
        ).generate_state(1)[0]
    )

    sim = simulate_metastable(
        seed
    )

    diag = metastable_diagnostics(
        sim
    )

    row = evaluate_methods(
        sim["subjects"],
        sim["true_patterns"],
        K_fit=N_TRUE_MOTIFS,
        seed=seed,
        regime=regime,
        rep=rep,
        true_activations=sim[
            "activations"
        ],
        truth_kind=sim[
            "truth_kind"
        ],
        extra={
            "condition": condition,
            **diag,
        },
    )

    results_df = save_row(
        results_df,
        row,
    )

print(
    "Core PCA-vs-MS-AA benchmark complete."
)


## 7. Core summary


In [ ]:
regime_order = [
    "PCA_control",
    "AA_control",
    "metastable",
]

regime_labels = [
    "PCA-favorable",
    "AA-favorable",
    "Metastable",
]

core = results_df[
    results_df["regime"].isin(
        regime_order
    )
].copy()

compact = (
    core
    .groupby("regime")
    .agg(
        n=("rep", "count"),

        AA_spatial=(
            "aa_motif_recovery",
            "mean",
        ),
        PCA_spatial=(
            "pca_motif_recovery",
            "mean",
        ),

        AA_temporal=(
            "aa_temporal_recovery",
            "mean",
        ),
        PCA_temporal=(
            "pca_temporal_recovery",
            "mean",
        ),

        AA_subnetwork=(
            "aa_subnetwork_recovery",
            "mean",
        ),
        PCA_subnetwork=(
            "pca_subnetwork_recovery",
            "mean",
        ),

        AA_span=(
            "aa_span_recovery",
            "mean",
        ),
        PCA_span=(
            "pca_span_recovery",
            "mean",
        ),

        AA_MSE=(
            "aa_mse",
            "mean",
        ),
        PCA_MSE=(
            "pca_mse",
            "mean",
        ),
    )
)

display(compact)


## 8. Direct pattern recovery and span recovery


In [ ]:
x = np.arange(
    len(regime_order)
)

width = 0.34

fig, ax = plt.subplots(
    figsize=(8.5, 4.8)
)

for j, (label, col) in enumerate([
    (
        "MS-AA",
        "aa_motif_recovery",
    ),
    (
        "PCA",
        "pca_motif_recovery",
    ),
]):
    means = []
    sems = []

    for regime in regime_order:
        vals = core.loc[
            core["regime"] == regime,
            col,
        ]

        means.append(
            vals.mean()
        )
        sems.append(
            vals.sem()
        )

    offset = (
        j - 0.5
    ) * width

    ax.bar(
        x + offset,
        means,
        width,
        yerr=sems,
        label=label,
        capsize=3,
    )

ax.set_xticks(x)
ax.set_xticklabels(
    regime_labels
)
ax.set_ylim(0, 1.05)
ax.set_ylabel(
    "One-to-one true-pattern recovery"
)
ax.set_title(
    "Direct recovery of latent spatial configurations"
)
ax.legend()

plt.show()


fig, ax = plt.subplots(
    figsize=(8.5, 4.8)
)

for j, (label, col) in enumerate([
    (
        "MS-AA",
        "aa_span_recovery",
    ),
    (
        "PCA",
        "pca_span_recovery",
    ),
]):
    means = []
    sems = []

    for regime in regime_order:
        vals = core.loc[
            core["regime"] == regime,
            col,
        ]

        means.append(
            vals.mean()
        )
        sems.append(
            vals.sem()
        )

    offset = (
        j - 0.5
    ) * width

    ax.bar(
        x + offset,
        means,
        width,
        yerr=sems,
        label=label,
        capsize=3,
    )

ax.set_xticks(x)
ax.set_xticklabels(
    regime_labels
)
ax.set_ylim(0, 1.05)
ax.set_ylabel(
    "Recovery from full recovered span"
)
ax.set_title(
    "Pattern identity versus low-dimensional span"
)
ax.legend()

plt.show()


# PART B — Paired inference

Inference is paired by simulation replicate. The primary inferential quantity
is the replicate-level difference **MS-AA − PCA**, summarized with a paired
bootstrap confidence interval. The sign-flip \(p\)-value is secondary.


In [ ]:
PAIR_BOOT_REPS = 20000
PAIR_PERM_REPS = 20000
PAIR_RNG_SEED = 20260908


def bootstrap_mean_ci(
    values,
    n_boot=PAIR_BOOT_REPS,
    seed=PAIR_RNG_SEED,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return (
            np.nan,
            np.nan,
            np.nan,
        )

    rng = np.random.default_rng(seed)

    idx = rng.integers(
        0,
        len(values),
        size=(
            n_boot,
            len(values),
        ),
    )

    boot_means = values[
        idx
    ].mean(
        axis=1
    )

    lo, hi = np.quantile(
        boot_means,
        [0.025, 0.975],
    )

    return (
        float(
            values.mean()
        ),
        float(lo),
        float(hi),
    )


def signflip_pvalue(
    values,
    n_perm=PAIR_PERM_REPS,
    seed=PAIR_RNG_SEED,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return np.nan

    observed = abs(
        values.mean()
    )

    rng = np.random.default_rng(
        seed
    )

    signs = rng.choice(
        [-1.0, 1.0],
        size=(
            n_perm,
            len(values),
        ),
    )

    perm_means = np.abs(
        (
            signs
            * values[None, :]
        ).mean(
            axis=1
        )
    )

    p = (
        np.sum(
            perm_means >= observed
        )
        + 1
    ) / (
        n_perm + 1
    )

    return float(p)


paired_rows = []

for r_i, regime in enumerate(
    regime_order
):
    sub = (
        core[
            core["regime"] == regime
        ]
        .sort_values("rep")
        .copy()
    )

    delta = (
        sub[
            "aa_motif_recovery"
        ].to_numpy()
        - sub[
            "pca_motif_recovery"
        ].to_numpy()
    )

    mean, lo, hi = (
        bootstrap_mean_ci(
            delta,
            seed=(
                PAIR_RNG_SEED
                + 100 * r_i
            ),
        )
    )

    paired_rows.append({
        "regime": regime,
        "comparison": "AA - PCA",
        "n": len(delta),
        "mean_delta": mean,
        "CI_low": lo,
        "CI_high": hi,
        "signflip_p": (
            signflip_pvalue(
                delta,
                seed=(
                    PAIR_RNG_SEED
                    + 1000
                    + 100 * r_i
                ),
            )
        ),
        "fraction_positive": float(
            np.mean(
                delta > 0
            )
        ),
    })

paired_inference_df = pd.DataFrame(
    paired_rows
)

display(
    paired_inference_df
)

meta_pair = paired_inference_df[
    paired_inference_df["regime"]
    == "metastable"
].copy()

if len(meta_pair):
    row = meta_pair.iloc[0]

    fig, ax = plt.subplots(
        figsize=(5.5, 4.2)
    )

    mean = row[
        "mean_delta"
    ]

    lo = row[
        "CI_low"
    ]

    hi = row[
        "CI_high"
    ]

    ax.errorbar(
        [0],
        [mean],
        yerr=np.array(
            [
                [mean - lo],
                [hi - mean],
            ]
        ),
        fmt="o",
        capsize=5,
    )

    ax.axhline(
        0,
        linestyle="--",
        linewidth=1,
    )

    ax.set_xticks([0])
    ax.set_xticklabels(
        ["MS-AA − PCA"]
    )

    ax.set_ylabel(
        "Paired difference in motif recovery"
    )

    ax.set_title(
        "Brain-inspired metastable regime"
    )

    plt.show()


# PART C — Metastable robustness to noise


In [ ]:
results_df = load_results()

for noise_sd in NOISE_LEVELS:
    for rep in range(
        N_REPS_ROBUST
    ):
        regime = "metastable_noise"
        condition = (
            f"noise_{noise_sd}"
        )

        if is_complete(
            results_df,
            regime,
            condition,
            rep,
        ):
            print(
                f"SKIP {condition} rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [
                    RNG_SEED,
                    404,
                    int(
                        noise_sd
                        * 1000
                    ),
                    rep,
                ]
            ).generate_state(1)[0]
        )

        sim = simulate_metastable(
            seed,
            noise_sd=noise_sd,
            heterogeneity_scale=1.0,
        )

        row = evaluate_methods(
            sim["subjects"],
            sim["true_patterns"],
            K_fit=N_TRUE_MOTIFS,
            seed=seed,
            regime=regime,
            rep=rep,
            true_activations=sim[
                "activations"
            ],
            truth_kind=sim[
                "truth_kind"
            ],
            extra={
                "condition": condition,
                "noise_sd": noise_sd,
                **metastable_diagnostics(
                    sim
                ),
            },
        )

        results_df = save_row(
            results_df,
            row,
        )

print(
    "Noise robustness complete."
)

noise_df = results_df[
    results_df["regime"]
    == "metastable_noise"
].copy()

if len(noise_df):
    display(
        noise_df
        .groupby("noise_sd")[
            [
                "aa_motif_recovery",
                "pca_motif_recovery",
                "aa_temporal_recovery",
                "pca_temporal_recovery",
            ]
        ]
        .agg(
            ["mean", "sem"]
        )
    )


# PART D — Metastable robustness to subject heterogeneity


In [ ]:
results_df = load_results()

for hetero in HETEROGENEITY_LEVELS:
    for rep in range(
        N_REPS_ROBUST
    ):
        regime = (
            "metastable_heterogeneity"
        )
        condition = (
            f"heterogeneity_{hetero}"
        )

        if is_complete(
            results_df,
            regime,
            condition,
            rep,
        ):
            print(
                f"SKIP {condition} rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [
                    RNG_SEED,
                    505,
                    int(
                        hetero
                        * 1000
                    ),
                    rep,
                ]
            ).generate_state(1)[0]
        )

        sim = simulate_metastable(
            seed,
            noise_sd=BASE_NOISE_SD,
            heterogeneity_scale=hetero,
        )

        row = evaluate_methods(
            sim["subjects"],
            sim["true_patterns"],
            K_fit=N_TRUE_MOTIFS,
            seed=seed,
            regime=regime,
            rep=rep,
            true_activations=sim[
                "activations"
            ],
            truth_kind=sim[
                "truth_kind"
            ],
            extra={
                "condition": condition,
                "heterogeneity_scale": hetero,
                **metastable_diagnostics(
                    sim
                ),
            },
        )

        results_df = save_row(
            results_df,
            row,
        )

print(
    "Heterogeneity robustness complete."
)

hetero_df = results_df[
    results_df["regime"]
    == "metastable_heterogeneity"
].copy()

if len(hetero_df):
    display(
        hetero_df
        .groupby(
            "heterogeneity_scale"
        )[
            [
                "aa_motif_recovery",
                "pca_motif_recovery",
                "aa_temporal_recovery",
                "pca_temporal_recovery",
            ]
        ]
        .agg(
            ["mean", "sem"]
        )
    )


# PART E — Metastable sensitivity to fitted K


In [ ]:
results_df = load_results()

for K_fit in K_LEVELS:
    for rep in range(
        N_REPS_ROBUST
    ):
        regime = "metastable_K"

        condition = (
            f"K_{K_fit}"
        )

        if is_complete(
            results_df,
            regime,
            condition,
            rep,
        ):
            print(
                f"SKIP {condition} "
                f"rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [
                    RNG_SEED,
                    606,
                    int(K_fit),
                    rep,
                ]
            ).generate_state(1)[0]
        )

        sim = (
            simulate_metastable(
                seed
            )
        )

        diag = (
            metastable_diagnostics(
                sim
            )
        )

        row = evaluate_methods(
            sim["subjects"],
            sim["true_patterns"],
            K_fit=K_fit,
            seed=seed,
            regime=regime,
            rep=rep,
            true_activations=(
                sim["activations"]
            ),
            truth_kind=(
                sim["truth_kind"]
            ),
            extra={
                "condition": (
                    condition
                ),
                **diag,
            },
        )

        results_df = save_row(
            results_df,
            row,
        )

print(
    "K sensitivity complete."
)


## K-sensitivity summary

For \(K<K_{\mathrm{true}}\), use the **coverage-adjusted** score as the primary
quantity. The ordinary matched mean is retained only as a diagnostic.

Temporal recovery also receives a coverage-adjusted version so that recovering
only a subset of the six latent configurations cannot look artificially
strong.


In [ ]:
k_df = results_df[
    results_df["regime"]
    == "metastable_K"
].copy()

if len(k_df):
    k_summary = (
        k_df
        .groupby("K_fit")[
            [
                "aa_motif_recovery_coverage",
                "pca_motif_recovery_coverage",
                "aa_temporal_recovery_coverage",
                "pca_temporal_recovery_coverage",
            ]
        ]
        .agg(
            ["mean", "sem"]
        )
    )

    display(k_summary)

    fig, ax = plt.subplots(
        figsize=(7.5, 4.5)
    )

    for label, col in [
        (
            "MS-AA",
            "aa_motif_recovery_coverage",
        ),
        (
            "PCA",
            "pca_motif_recovery_coverage",
        ),
    ]:
        means = (
            k_df
            .groupby(
                "K_fit"
            )[col]
            .mean()
        )

        sems = (
            k_df
            .groupby(
                "K_fit"
            )[col]
            .sem()
        )

        ax.errorbar(
            means.index,
            means.values,
            yerr=sems.values,
            marker="o",
            label=label,
        )

    ax.axvline(
        N_TRUE_MOTIFS,
        linestyle="--",
        linewidth=1,
    )

    ax.set_xlabel(
        "Fitted K"
    )

    ax.set_ylabel(
        "Coverage-adjusted spatial recovery"
    )

    ax.set_title(
        "Metastable sensitivity to fitted model order"
    )

    ax.legend()

    plt.show()


# PART F — Metastable sensitivity to motif overlap

The same metastable dynamics are generated from low-, baseline-, and
high-overlap versions of the distributed motifs. The mean pairwise cosine
similarity of the true motifs is printed so the manipulation is directly
verifiable.


In [ ]:
results_df = load_results()

for (
    overlap_name,
    distinctiveness_scale,
) in OVERLAP_LEVELS.items():

    motifs_variant = (
        make_overlap_variant(
            TRUE_MOTIFS_RAW,
            distinctiveness_scale,
        )
    )

    overlap_cosine = (
        mean_pairwise_cosine(
            motifs_variant
        )
    )

    print(
        overlap_name,
        "| mean pairwise cosine =",
        round(
            overlap_cosine,
            3,
        ),
    )

    for rep in range(
        N_REPS_SENSITIVITY
    ):
        regime = (
            "metastable_overlap"
        )

        condition = (
            overlap_name
        )

        if is_complete(
            results_df,
            regime,
            condition,
            rep,
        ):
            print(
                f"SKIP {condition} "
                f"rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [
                    RNG_SEED,
                    707,
                    list(
                        OVERLAP_LEVELS
                    ).index(
                        overlap_name
                    ),
                    rep,
                ]
            ).generate_state(1)[0]
        )

        sim = (
            simulate_metastable(
                seed,
                motifs_raw=(
                    motifs_variant
                ),
            )
        )

        diag = (
            metastable_diagnostics(
                sim
            )
        )

        row = evaluate_methods(
            sim["subjects"],
            sim["true_patterns"],
            K_fit=N_TRUE_MOTIFS,
            seed=seed,
            regime=regime,
            rep=rep,
            true_activations=(
                sim["activations"]
            ),
            truth_kind=(
                sim["truth_kind"]
            ),
            extra={
                "condition": (
                    condition
                ),
                "distinctiveness_scale": (
                    distinctiveness_scale
                ),
                "mean_pairwise_cosine": (
                    overlap_cosine
                ),
                **diag,
            },
        )

        results_df = save_row(
            results_df,
            row,
        )

print(
    "Overlap sensitivity complete."
)


In [ ]:
overlap_df = results_df[
    results_df["regime"]
    == "metastable_overlap"
].copy()

if len(overlap_df):
    display(
        overlap_df
        .groupby(
            "condition"
        )[
            [
                "mean_pairwise_cosine",
                "aa_motif_recovery",
                "pca_motif_recovery",
                "aa_temporal_recovery",
                "pca_temporal_recovery",
            ]
        ]
        .agg(
            ["mean", "sem"]
        )
    )

    order = [
        "low_overlap",
        "baseline_overlap",
        "high_overlap",
    ]

    fig, ax = plt.subplots(
        figsize=(7.5, 4.5)
    )

    for label, col in [
        (
            "MS-AA",
            "aa_motif_recovery",
        ),
        (
            "PCA",
            "pca_motif_recovery",
        ),
    ]:
        means = [
            overlap_df.loc[
                overlap_df[
                    "condition"
                ] == cond,
                col,
            ].mean()
            for cond in order
        ]

        sems = [
            overlap_df.loc[
                overlap_df[
                    "condition"
                ] == cond,
                col,
            ].sem()
            for cond in order
        ]

        ax.errorbar(
            np.arange(
                len(order)
            ),
            means,
            yerr=sems,
            marker="o",
            label=label,
        )

    ax.set_xticks(
        np.arange(
            len(order)
        )
    )

    ax.set_xticklabels(
        [
            "Low",
            "Baseline",
            "High",
        ]
    )

    ax.set_xlabel(
        "Motif overlap"
    )

    ax.set_ylabel(
        "Spatial motif recovery"
    )

    ax.set_title(
        "Sensitivity to latent motif overlap"
    )

    ax.legend()

    plt.show()


# PART G — Metastable sensitivity to dynamical mixing

This is a small, interpretable sensitivity analysis rather than a factorial
sweep:

- **state-like**: longer dwell times, sharper transitions, less secondary
  recruitment;
- **baseline**: manuscript default;
- **more mixed**: shorter dwell times, broader transitions, stronger secondary
  recruitment, and more persistent background activity.

The useful question is whether an MS-AA advantage persists or diminishes as
the process becomes less discrete and more continuously blended.


In [ ]:
results_df = load_results()

for (
    dynamics_name,
    pars,
) in DYNAMICS_PRESETS.items():

    for rep in range(
        N_REPS_SENSITIVITY
    ):
        regime = (
            "metastable_dynamics"
        )

        condition = (
            dynamics_name
        )

        if is_complete(
            results_df,
            regime,
            condition,
            rep,
        ):
            print(
                f"SKIP {condition} "
                f"rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [
                    RNG_SEED,
                    808,
                    list(
                        DYNAMICS_PRESETS
                    ).index(
                        dynamics_name
                    ),
                    rep,
                ]
            ).generate_state(1)[0]
        )

        sim = simulate_metastable(
            seed,
            mean_dwell=(
                pars[
                    "mean_dwell"
                ]
            ),
            transition_width=(
                pars[
                    "transition_width"
                ]
            ),
            secondary_weight=(
                pars[
                    "secondary_weight"
                ]
            ),
            ar1_rho=(
                pars[
                    "ar1_rho"
                ]
            ),
        )

        diag = (
            metastable_diagnostics(
                sim
            )
        )

        row = evaluate_methods(
            sim["subjects"],
            sim["true_patterns"],
            K_fit=N_TRUE_MOTIFS,
            seed=seed,
            regime=regime,
            rep=rep,
            true_activations=(
                sim["activations"]
            ),
            truth_kind=(
                sim["truth_kind"]
            ),
            extra={
                "condition": (
                    condition
                ),
                **pars,
                **diag,
            },
        )

        results_df = save_row(
            results_df,
            row,
        )

print(
    "Dynamics sensitivity complete."
)


In [ ]:
dynamics_df = results_df[
    results_df["regime"]
    == "metastable_dynamics"
].copy()

if len(dynamics_df):
    display(
        dynamics_df
        .groupby(
            "condition"
        )[
            [
                "mean_dwell",
                "mean_activation_lag1",
                "aa_motif_recovery",
                "pca_motif_recovery",
                "aa_temporal_recovery",
                "pca_temporal_recovery",
            ]
        ]
        .agg(
            ["mean", "sem"]
        )
    )

    order = [
        "state_like",
        "baseline",
        "more_mixed",
    ]

    fig, ax = plt.subplots(
        figsize=(7.5, 4.5)
    )

    for label, col in [
        (
            "MS-AA",
            "aa_motif_recovery",
        ),
        (
            "PCA",
            "pca_motif_recovery",
        ),
    ]:
        means = [
            dynamics_df.loc[
                dynamics_df[
                    "condition"
                ] == cond,
                col,
            ].mean()
            for cond in order
        ]

        sems = [
            dynamics_df.loc[
                dynamics_df[
                    "condition"
                ] == cond,
                col,
            ].sem()
            for cond in order
        ]

        ax.errorbar(
            np.arange(
                len(order)
            ),
            means,
            yerr=sems,
            marker="o",
            label=label,
        )

    ax.set_xticks(
        np.arange(
            len(order)
        )
    )

    ax.set_xticklabels(
        [
            "State-like",
            "Baseline",
            "More mixed",
        ]
    )

    ax.set_ylabel(
        "Spatial motif recovery"
    )

    ax.set_title(
        "Sensitivity to dynamical mixing"
    )

    ax.legend()

    plt.show()


# PART H — Spatial versus temporal recovery

Spatial recovery asks **what** configuration was recovered. Temporal recovery
asks whether the **same spatially matched dimension** recovered **when** that
configuration was expressed.

PCA temporal sign is ignored because component sign is arbitrary. MS-AA is
evaluated without sign reversal.


In [ ]:
core = results_df[
    results_df["regime"].isin(
        regime_order
    )
].copy()

temporal_summary = (
    core
    .groupby("regime")[
        [
            "aa_temporal_recovery",
            "pca_temporal_recovery",
        ]
    ]
    .agg(
        ["mean", "sem"]
    )
)

display(
    temporal_summary
)

meta_temporal = core[
    core["regime"]
    == "metastable"
].copy()

delta = (
    meta_temporal[
        "aa_temporal_recovery"
    ].to_numpy()
    - meta_temporal[
        "pca_temporal_recovery"
    ].to_numpy()
)

mean, lo, hi = bootstrap_mean_ci(
    delta,
    seed=PAIR_RNG_SEED + 3000,
)

temporal_pair_df = pd.DataFrame([
    {
        "comparison": "AA - PCA",
        "n": len(delta),
        "mean_delta": mean,
        "CI_low": lo,
        "CI_high": hi,
        "signflip_p": (
            signflip_pvalue(
                delta,
                seed=(
                    PAIR_RNG_SEED
                    + 4000
                ),
            )
        ),
        "fraction_positive": float(
            np.mean(
                delta > 0
            )
        ),
    }
])

display(
    temporal_pair_df
)


# PART I — Normalization QA

This compares the simulation fallback normalization with the exact empirical
`hypertools/normalize.py` implementation when available. The already-verified
difference in the quick run was on the order of \(10^{-7}\), so this cell is a
reproducibility check rather than an unresolved analysis.


In [ ]:
def load_empirical_normalize_module(path):
    """
    Load the empirical hypertools/normalize.py module even though it uses
    local imports such as:

        from format_data import format_data as formatter
        from helpers import memoize

    Loading normalize.py directly with spec_from_file_location() does not
    automatically put its containing directory on sys.path. We therefore
    expose that directory temporarily while the module and its local
    dependencies are imported.
    """
    path = Path(path).expanduser().resolve()

    if not path.exists():
        raise FileNotFoundError(
            f"Empirical normalize.py not found: {path}"
        )

    module_dir = str(path.parent)
    added_path = module_dir not in sys.path

    if added_path:
        sys.path.insert(
            0,
            module_dir,
        )

    try:
        spec = importlib.util.spec_from_file_location(
            "empirical_normalize",
            str(path),
        )

        if spec is None or spec.loader is None:
            raise ImportError(
                f"Could not create import spec for {path}"
            )

        ht = importlib.util.module_from_spec(
            spec
        )

        spec.loader.exec_module(
            ht
        )

    finally:
        if added_path:
            try:
                sys.path.remove(
                    module_dir
                )
            except ValueError:
                pass

    return ht


def normalization_qa():
    """
    Compare the simulation fallback 'across' normalization with the exact
    empirical hypertools implementation on the same simulated subject data.

    This QA does not change the simulation results automatically. It reports
    the maximum absolute discrepancy so that we can decide whether the
    simulation normalizer needs to be replaced by the empirical routine.
    """
    qa_sim = simulate_metastable(
        123456
    )

    raw = qa_sim[
        "subjects"
    ][:3]

    fallback, _, _ = (
        normalize_across_subjects(
            raw
        )
    )

    normalize_path = (
        Path(
            HYPERTOOLS_NORMALIZE_PATH
        )
        .expanduser()
        .resolve()
    )

    if not normalize_path.exists():
        print(
            "Exact hypertools normalizer not found at:",
            normalize_path,
        )
        print(
            "Normalization QA skipped; this does not affect the simulation run."
        )
        return None

    ht = load_empirical_normalize_module(
        normalize_path
    )

    coerced = [
        coerce_TxV(x)
        for x in raw
    ]

    # We have already converted the inputs to numeric T x V arrays.
    # Disable HyperTools' separate format_data() layer here; otherwise
    # normalize() tries to import optional legacy dependencies such as
    # df2mat/text2mat/datageometry that are irrelevant to this QA.
    exact = ht.normalize(
        coerced,
        normalize="across",
        format_data=False,
    )

    diffs = [
        np.max(
            np.abs(
                np.asarray(
                    a,
                    dtype=float,
                )
                - np.asarray(
                    b,
                    dtype=float,
                )
            )
        )
        for a, b in zip(
            fallback,
            exact,
        )
    ]

    max_diff = float(
        np.max(diffs)
    )

    print(
        "Maximum absolute fallback-vs-exact difference:",
        max_diff,
    )

    if np.isfinite(max_diff):
        if max_diff < 1e-10:
            print(
                "Normalization QA: PASS — fallback and empirical "
                "normalization agree to numerical precision."
            )
        elif max_diff < 1e-6:
            print(
                "Normalization QA: very small numerical discrepancy."
            )
        else:
            print(
                "Normalization QA: substantive discrepancy detected. "
                "Inspect the empirical normalization definition before "
                "treating the simulation normalization as pipeline-matched."
            )

    return max_diff


NORMALIZATION_QA_MAX_DIFF = (
    normalization_qa()
)


# PART J — Representative metastable spatial and temporal recovery


In [ ]:
REPRESENTATIVE_SEED = 20260908

sim = simulate_metastable(
    REPRESENTATIVE_SEED
)

(
    subjects,
    mean_subject_mean,
    sd,
) = normalize_across_subjects(
    sim["subjects"]
)

true_patterns = transform_true_patterns(
    sim["true_patterns"],
    mean_subject_mean,
    sd,
    truth_kind=sim[
        "truth_kind"
    ],
)

X_stack = np.vstack(
    subjects
)

# ---------- MS-AA ----------
msaa = fit_msaa(
    subjects,
    K=N_TRUE_MOTIFS,
    analysis_type="spatial",
    seed=REPRESENTATIVE_SEED,
    verbose=True,
)

S_group = np.mean(
    np.stack(
        [
            np.asarray(
                sub["S"],
                dtype=float,
            )
            for sub in msaa[
                "results_subj"
            ]
        ],
        axis=0,
    ),
    axis=0,
)

aa_scores = [
    np.asarray(
        sub["sXC"],
        dtype=float,
    )
    for sub in msaa[
        "results_subj"
    ]
]

# ---------- PCA ----------
pca = PCA(
    n_components=N_TRUE_MOTIFS,
    random_state=REPRESENTATIVE_SEED,
).fit(
    X_stack
)

pca_patterns = np.asarray(
    pca.components_,
    dtype=float,
)

pca_scores = pca.transform(
    X_stack
)

# ---------- Spatial matching ----------
aa_match = optimal_match(
    true_patterns,
    S_group,
    absolute=False,
)

pca_match = optimal_match(
    true_patterns,
    pca_patterns,
    absolute=True,
)

AA_ORDERED = S_group[
    aa_match[
        "recovered_idx"
    ]
].copy()

PCA_ORDERED = pca_patterns[
    pca_match[
        "recovered_idx"
    ]
].copy()

TRUE_ORDERED = true_patterns[
    aa_match[
        "true_idx"
    ]
].copy()

# Orient PCA only for visualization.
for k, true_idx in enumerate(
    pca_match["true_idx"]
):
    rec_idx = pca_match[
        "recovered_idx"
    ][k]

    r = np.corrcoef(
        true_patterns[
            true_idx
        ],
        pca_patterns[
            rec_idx
        ],
    )[0, 1]

    if np.isfinite(r) and r < 0:
        PCA_ORDERED[k] *= -1


plot_pattern_matrix(
    TRUE_ORDERED,
    "Representative metastable simulation: true configurations",
    ylabel="True configuration",
)

plot_pattern_matrix(
    AA_ORDERED,
    "Representative metastable simulation: matched MS-AA archetypes",
    ylabel="Matched archetype",
)

plot_pattern_matrix(
    PCA_ORDERED,
    "Representative metastable simulation: matched PCA components",
    ylabel="Matched PCA component",
)


aa_temp = temporal_recovery_from_spatial_match(
    sim["activations"],
    aa_scores,
    aa_match,
    absolute=False,
)

pca_temp = temporal_recovery_from_spatial_match(
    sim["activations"],
    pca_scores,
    pca_match,
    absolute=True,
)

print(
    "Representative spatial recovery:",
    {
        "MS-AA": float(
            np.mean(
                aa_match["matched"]
            )
        ),
        "PCA": float(
            np.mean(
                pca_match["matched"]
            )
        ),
    },
)

print(
    "Representative temporal recovery:",
    {
        "MS-AA": aa_temp["mean"],
        "PCA": pca_temp["mean"],
    },
)


# PART K — Final manuscript-oriented summary


In [ ]:
results_df = load_results()

core = results_df[
    results_df["regime"].isin(
        regime_order
    )
].copy()

final_rows = []

for regime in regime_order:
    sub = core[
        core["regime"]
        == regime
    ]

    if len(sub) == 0:
        continue

    final_rows.append({
        "regime": regime,
        "n": len(sub),

        "MS-AA spatial": (
            sub[
                "aa_motif_recovery"
            ].mean()
        ),
        "PCA spatial": (
            sub[
                "pca_motif_recovery"
            ].mean()
        ),

        "MS-AA temporal": (
            sub[
                "aa_temporal_recovery"
            ].mean()
        ),
        "PCA temporal": (
            sub[
                "pca_temporal_recovery"
            ].mean()
        ),

        "MS-AA subnetwork": (
            sub[
                "aa_subnetwork_recovery"
            ].mean()
        ),
        "PCA subnetwork": (
            sub[
                "pca_subnetwork_recovery"
            ].mean()
        ),

        "MS-AA span": (
            sub[
                "aa_span_recovery"
            ].mean()
        ),
        "PCA span": (
            sub[
                "pca_span_recovery"
            ].mean()
        ),

        "MS-AA MSE": (
            sub[
                "aa_mse"
            ].mean()
        ),
        "PCA MSE": (
            sub[
                "pca_mse"
            ].mean()
        ),
    })

final_summary_df = pd.DataFrame(
    final_rows
)

display(
    final_summary_df
)


# Interpretation guide

A well-calibrated benchmark should show **method-specific reversals**, not
universal superiority.

### PCA-favorable control
PCA should recover the true covariance directions best.

### AA-favorable control
MS-AA should recover the extremal convex-mixture motifs best.

### Brain-inspired metastable regime
This is the scientific test. The generator is deliberately not an exact PCA
or AA model. The key distinction is between:

- **individual configuration recovery**, where an interpretable extremal basis
  may have an advantage;
- **span recovery / reconstruction**, where PCA may preserve the same
  low-dimensional information as well as or better than MS-AA.

The simulation therefore should not be used to claim that neural activity is
literally simplex-organized or that archetypes are biological attractors.
Rather, it asks whether a plausible recurring-network geometry creates a
setting in which extremal representations expose individual distributed
configurations more directly than variance-based axes.


# OPTIONAL SUPPLEMENT — ICA positive-control calibration

This section is intentionally isolated from the main paper analysis. It is
**off by default** and does not alter the main results file.

If enabled, it runs a small independent-source positive control and compares
MS-AA, PCA, and ICA. Its purpose is only to demonstrate that the simulation
framework is capable of favoring ICA when ICA's assumptions are actually the
generating assumptions.


In [ ]:
# ============================================================
# OPTIONAL SUPPLEMENTARY ICA CALIBRATION
# ============================================================
#
# This cell is deliberately isolated from the primary PCA-vs-MS-AA
# benchmark. It does not write to the main checkpoint and is OFF by default.
#
# Its only purpose is a small methodological sanity check:
# on data generated from independent non-Gaussian sources, ICA should recover
# the generating mixing patterns more directly than PCA or MS-AA.
#
# Set RUN_OPTIONAL_ICA_CALIBRATION = True in the configuration cell to run.

if not RUN_OPTIONAL_ICA_CALIBRATION:
    print(
        "Optional ICA calibration skipped. "
        "Set RUN_OPTIONAL_ICA_CALIBRATION = True to run it."
    )

else:
    import warnings

    from sklearn.decomposition import FastICA
    from sklearn.exceptions import ConvergenceWarning

    def make_ica_mixing_patterns(
        seed,
    ):
        rng = np.random.default_rng(
            seed
        )

        base = rng.normal(
            size=(
                N_TRUE_MOTIFS,
                V,
            )
        )

        shared = rng.normal(
            size=(1, V)
        )

        shared /= (
            np.linalg.norm(
                shared
            )
            + 1e-12
        )

        patterns = (
            base
            + 0.45 * shared
        )

        patterns -= patterns.mean(
            axis=1,
            keepdims=True,
        )

        patterns /= (
            np.linalg.norm(
                patterns,
                axis=1,
                keepdims=True,
            )
            + 1e-12
        )

        return patterns


    def simulate_ica_control(
        seed,
    ):
        rng = np.random.default_rng(
            seed
        )

        true_patterns = (
            make_ica_mixing_patterns(
                seed + 999
            )
        )

        subjects = []
        activations = []

        for s in range(
            N_SUBJECTS
        ):
            sources = rng.laplace(
                loc=0.0,
                scale=(
                    1.0
                    / np.sqrt(2.0)
                ),
                size=(
                    T_PER_SUBJECT,
                    N_TRUE_MOTIFS,
                ),
            )

            X = (
                sources
                @ true_patterns
            )

            X += rng.normal(
                0,
                BASE_NOISE_SD,
                size=X.shape,
            )

            subjects.append(X)
            activations.append(
                sources
            )

        return {
            "subjects": subjects,
            "true_patterns": (
                true_patterns
            ),
            "activations": (
                activations
            ),
            "truth_kind": "direction",
        }


    def fit_fastica_once(
        X_stack,
        K_fit,
        seed,
    ):
        ica = FastICA(
            n_components=int(
                K_fit
            ),
            whiten="unit-variance",
            algorithm="parallel",
            fun="logcosh",
            max_iter=3000,
            tol=1e-5,
            random_state=int(seed),
        )

        with warnings.catch_warnings(
            record=True
        ) as caught:
            warnings.simplefilter(
                "always",
                ConvergenceWarning,
            )

            scores = ica.fit_transform(
                X_stack
            )

        converged = not any(
            issubclass(
                w.category,
                ConvergenceWarning,
            )
            for w in caught
        )

        return {
            "patterns": np.asarray(
                ica.mixing_,
                dtype=float,
            ).T,
            "scores": scores,
            "converged": bool(
                converged
            ),
        }


    rows = []

    for rep in range(
        OPTIONAL_ICA_REPS
    ):
        seed = int(
            np.random.SeedSequence(
                [
                    RNG_SEED,
                    909,
                    rep,
                ]
            ).generate_state(1)[0]
        )

        sim = simulate_ica_control(
            seed
        )

        # Reuse the primary evaluator for MS-AA and PCA.
        base = evaluate_methods(
            sim["subjects"],
            sim["true_patterns"],
            K_fit=N_TRUE_MOTIFS,
            seed=seed,
            regime="ICA_control_optional",
            rep=rep,
            true_activations=sim[
                "activations"
            ],
            truth_kind="direction",
        )

        (
            subjects,
            mean_subject_mean,
            sd,
        ) = normalize_across_subjects(
            sim["subjects"]
        )

        true_patterns = transform_true_patterns(
            sim["true_patterns"],
            mean_subject_mean,
            sd,
            truth_kind="direction",
        )

        X_stack = np.vstack(
            subjects
        )

        ica_fit = fit_fastica_once(
            X_stack,
            N_TRUE_MOTIFS,
            seed,
        )

        ica_match = optimal_match(
            true_patterns,
            ica_fit[
                "patterns"
            ],
            absolute=True,
        )

        ica_temporal = (
            temporal_recovery_from_spatial_match(
                sim[
                    "activations"
                ],
                ica_fit[
                    "scores"
                ],
                ica_match,
                absolute=True,
            )
        )

        rows.append({
            "rep": rep,
            "MS-AA spatial": (
                base[
                    "aa_motif_recovery"
                ]
            ),
            "PCA spatial": (
                base[
                    "pca_motif_recovery"
                ]
            ),
            "ICA spatial": float(
                np.mean(
                    ica_match[
                        "matched"
                    ]
                )
            ),
            "MS-AA temporal": (
                base[
                    "aa_temporal_recovery"
                ]
            ),
            "PCA temporal": (
                base[
                    "pca_temporal_recovery"
                ]
            ),
            "ICA temporal": (
                ica_temporal[
                    "mean"
                ]
            ),
            "ICA converged": (
                ica_fit[
                    "converged"
                ]
            ),
        })

    optional_ica_df = pd.DataFrame(
        rows
    )

    display(
        optional_ica_df
    )

    display(
        optional_ica_df.mean(
            numeric_only=True
        ).to_frame(
            "mean"
        )
    )


In [ ]:
# ============================================================
# PUBLICATION FIGURE HELPERS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA


FIGURE_BOOT_REPS = 20000
FIGURE_RNG_SEED = 20260909


def bootstrap_ci(
    values,
    n_boot=FIGURE_BOOT_REPS,
    seed=FIGURE_RNG_SEED,
    ci=95,
):
    """
    Percentile bootstrap CI for the mean.
    """
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return (
            np.nan,
            np.nan,
            np.nan,
        )

    rng = np.random.default_rng(
        seed
    )

    idx = rng.integers(
        0,
        len(values),
        size=(
            n_boot,
            len(values),
        ),
    )

    boot = values[
        idx
    ].mean(
        axis=1
    )

    alpha = (
        100 - ci
    ) / 2

    lo, hi = np.percentile(
        boot,
        [
            alpha,
            100 - alpha,
        ],
    )

    return (
        float(
            values.mean()
        ),
        float(lo),
        float(hi),
    )


def errorbar_from_ci(
    mean,
    lo,
    hi,
):
    """
    Convert lower/upper CI to matplotlib yerr.
    """
    return np.array([
        [
            mean - lo
        ],
        [
            hi - mean
        ],
    ])


def row_standardize_for_plot(
    X,
):
    """
    Row-standardize patterns ONLY for heatmap visualization.

    Recovery statistics themselves continue to use the original
    normalized-space patterns.
    """
    X = np.asarray(
        X,
        dtype=float,
    )

    mu = X.mean(
        axis=1,
        keepdims=True,
    )

    sd = X.std(
        axis=1,
        keepdims=True,
    )

    sd[
        sd < 1e-12
    ] = 1.0

    return (
        X - mu
    ) / sd


def reorder_to_truth(
    true_patterns,
    recovered_patterns,
    match,
    orient_sign=False,
):
    """
    Put recovered patterns into true-pattern order.

    PCA directions can be sign-flipped for visualization because
    component sign is arbitrary.
    """
    true_patterns = np.asarray(
        true_patterns,
        dtype=float,
    )

    recovered_patterns = np.asarray(
        recovered_patterns,
        dtype=float,
    )

    ordered = np.full(
        true_patterns.shape,
        np.nan,
        dtype=float,
    )

    for (
        true_idx,
        rec_idx,
    ) in zip(
        match["true_idx"],
        match["recovered_idx"],
    ):

        vec = recovered_patterns[
            rec_idx
        ].copy()

        if orient_sign:

            r = np.corrcoef(
                true_patterns[
                    true_idx
                ],
                vec,
            )[0, 1]

            if (
                np.isfinite(r)
                and r < 0
            ):
                vec *= -1

        ordered[
            true_idx
        ] = vec

    return ordered


def add_network_boundaries(
    ax,
):
    """
    Mark simulated subnetwork/network boundaries on heatmaps.
    """
    total_nodes = (
        N_NETWORKS
        * SUBNETWORKS_PER_NETWORK
        * NODES_PER_SUBNETWORK
    )

    # Thin subnetwork boundaries
    for x in range(
        NODES_PER_SUBNETWORK,
        total_nodes,
        NODES_PER_SUBNETWORK,
    ):
        ax.axvline(
            x - 0.5,
            linewidth=0.35,
            alpha=0.35,
        )

    # Stronger network boundaries
    network_width = (
        SUBNETWORKS_PER_NETWORK
        * NODES_PER_SUBNETWORK
    )

    for x in range(
        network_width,
        total_nodes,
        network_width,
    ):
        ax.axvline(
            x - 0.5,
            linewidth=1.0,
            alpha=0.65,
        )


def objectively_select_representative_metastable(
    results_df,
):
    """
    Select the metastable replicate whose AA-PCA spatial-recovery
    difference is closest to the median paired difference.

    This avoids selecting the most visually favorable example.
    """
    meta = (
        results_df[
            results_df["regime"]
            == "metastable"
        ]
        .copy()
        .sort_values("rep")
    )

    meta[
        "aa_minus_pca"
    ] = (
        meta[
            "aa_motif_recovery"
        ]
        - meta[
            "pca_motif_recovery"
        ]
    )

    target = meta[
        "aa_minus_pca"
    ].median()

    idx = (
        meta[
            "aa_minus_pca"
        ]
        - target
    ).abs().idxmin()

    return (
        meta.loc[
            idx
        ].copy()
    )


def recover_representative_patterns(
    representative_row,
):
    """
    Re-run the objectively selected metastable simulation and
    recover spatial patterns for MS-AA and PCA.
    """
    seed = int(
        representative_row[
            "seed"
        ]
    )

    sim = simulate_metastable(
        seed
    )

    (
        subjects,
        mean_subject_mean,
        sd,
    ) = normalize_across_subjects(
        sim[
            "subjects"
        ]
    )

    true_patterns = (
        transform_true_patterns(
            sim[
                "true_patterns"
            ],
            mean_subject_mean,
            sd,
            truth_kind=sim[
                "truth_kind"
            ],
        )
    )

    X_stack = np.vstack(
        subjects
    )

    # ------------------------
    # MS-AA
    # ------------------------

    msaa = fit_msaa(
        subjects,
        K=N_TRUE_MOTIFS,
        analysis_type="spatial",
        seed=seed,
        verbose=False,
    )

    aa_patterns = np.mean(
        np.stack(
            [
                np.asarray(
                    sub["S"],
                    dtype=float,
                )
                for sub in msaa[
                    "results_subj"
                ]
            ],
            axis=0,
        ),
        axis=0,
    )

    aa_match = optimal_match(
        true_patterns,
        aa_patterns,
        absolute=False,
    )

    aa_ordered = reorder_to_truth(
        true_patterns,
        aa_patterns,
        aa_match,
        orient_sign=False,
    )

    # ------------------------
    # PCA
    # ------------------------

    pca = PCA(
        n_components=N_TRUE_MOTIFS,
        random_state=seed,
    )

    pca.fit(
        X_stack
    )

    pca_patterns = np.asarray(
        pca.components_,
        dtype=float,
    )

    pca_match = optimal_match(
        true_patterns,
        pca_patterns,
        absolute=True,
    )

    pca_ordered = reorder_to_truth(
        true_patterns,
        pca_patterns,
        pca_match,
        orient_sign=True,
    )

    return {
        "seed": seed,
        "true": true_patterns,
        "aa": aa_ordered,
        "pca": pca_ordered,
        "aa_match": aa_match,
        "pca_match": pca_match,
    }


print(
    "Publication plotting helpers loaded."
)

In [ ]:
# ============================================================
# MAIN SIMULATION FIGURE
# ============================================================

results_df = load_results()

core_order = [
    "PCA_control",
    "AA_control",
    "metastable",
]

core_labels = [
    "PCA-favorable",
    "AA-favorable",
    "Metastable",
]

core = results_df[
    results_df["regime"].isin(
        core_order
    )
].copy()


# ============================================================
# Objectively select representative metastable replicate
# ============================================================

representative_row = (
    objectively_select_representative_metastable(
        results_df
    )
)

representative = (
    recover_representative_patterns(
        representative_row
    )
)

print(
    "Representative metastable replicate:"
)

display(
    representative_row[
        [
            "rep",
            "seed",
            "aa_motif_recovery",
            "pca_motif_recovery",
        ]
    ]
)

print(
    "AA - PCA spatial difference:",
    representative_row[
        "aa_motif_recovery"
    ]
    - representative_row[
        "pca_motif_recovery"
    ],
)


# ============================================================
# Figure layout
# ============================================================

fig = plt.figure(
    figsize=(14, 9.5)
)

gs = fig.add_gridspec(
    2,
    3,
    height_ratios=[
        1.0,
        1.18,
    ],
    hspace=0.42,
    wspace=0.38,
)

axA = fig.add_subplot(
    gs[
        0,
        0,
    ]
)

axB = fig.add_subplot(
    gs[
        0,
        1,
    ]
)

axC = fig.add_subplot(
    gs[
        0,
        2,
    ]
)

heat_gs = (
    gs[
        1,
        :
    ]
    .subgridspec(
        1,
        3,
        wspace=0.08,
    )
)

heat_axes = [
    fig.add_subplot(
        heat_gs[
            0,
            i,
        ]
    )
    for i in range(3)
]


# ============================================================
# PANEL A
# Core benchmark: replicate distributions + bootstrap CIs
# ============================================================

method_specs = [
    (
        "MS-AA",
        "aa_motif_recovery",
    ),
    (
        "PCA",
        "pca_motif_recovery",
    ),
]

default_colors = (
    plt.rcParams[
        "axes.prop_cycle"
    ]
    .by_key()[
        "color"
    ]
)

offsets = [
    -0.11,
    0.11,
]

rng = np.random.default_rng(
    FIGURE_RNG_SEED
)

for method_i, (
    label,
    col,
) in enumerate(
    method_specs
):

    for regime_i, regime in enumerate(
        core_order
    ):

        vals = (
            core.loc[
                core[
                    "regime"
                ]
                == regime,
                col,
            ]
            .dropna()
            .to_numpy()
        )

        x_center = (
            regime_i
            + offsets[
                method_i
            ]
        )

        # Raw simulation replicates
        jitter = rng.normal(
            0,
            0.018,
            size=len(
                vals
            ),
        )

        axA.scatter(
            np.full(
                len(vals),
                x_center,
            )
            + jitter,
            vals,
            s=13,
            alpha=0.28,
            color=default_colors[
                method_i
            ],
        )

        mean, lo, hi = (
            bootstrap_ci(
                vals,
                seed=(
                    FIGURE_RNG_SEED
                    + 100
                    * regime_i
                    + method_i
                ),
            )
        )

        axA.errorbar(
            x_center,
            mean,
            yerr=errorbar_from_ci(
                mean,
                lo,
                hi,
            ),
            fmt="o",
            markersize=7,
            capsize=4,
            linewidth=1.8,
            color=default_colors[
                method_i
            ],
            label=(
                label
                if regime_i == 0
                else None
            ),
        )

axA.set_xticks(
    np.arange(
        len(
            core_order
        )
    )
)

axA.set_xticklabels(
    core_labels,
    rotation=18,
    ha="right",
)

axA.set_ylim(
    0,
    1.04,
)

axA.set_ylabel(
    "Spatial configuration recovery (r)"
)

axA.set_title(
    "A  Method-specific recovery",
    loc="left",
    fontweight="bold",
)

axA.legend(
    frameon=False
)

axA.spines[
    "top"
].set_visible(
    False
)

axA.spines[
    "right"
].set_visible(
    False
)


# ============================================================
# PANEL B
# Paired metastable replicate plot
# ============================================================

meta = (
    core[
        core[
            "regime"
        ]
        == "metastable"
    ]
    .copy()
    .sort_values(
        "rep"
    )
)

pca_vals = meta[
    "pca_motif_recovery"
].to_numpy()

aa_vals = meta[
    "aa_motif_recovery"
].to_numpy()

for pca_val, aa_val in zip(
    pca_vals,
    aa_vals,
):

    axB.plot(
        [
            0,
            1,
        ],
        [
            pca_val,
            aa_val,
        ],
        marker="o",
        markersize=3.5,
        linewidth=0.9,
        alpha=0.35,
        color=default_colors[
            2
        ],
    )


# Mean + bootstrap CI for each method

for x_pos, vals, method_i in [
    (
        0,
        pca_vals,
        1,
    ),
    (
        1,
        aa_vals,
        0,
    ),
]:

    mean, lo, hi = (
        bootstrap_ci(
            vals,
            seed=(
                FIGURE_RNG_SEED
                + 500
                + x_pos
            ),
        )
    )

    axB.errorbar(
        x_pos,
        mean,
        yerr=errorbar_from_ci(
            mean,
            lo,
            hi,
        ),
        fmt="o",
        markersize=8,
        capsize=5,
        linewidth=2,
        color=default_colors[
            method_i
        ],
    )


delta = (
    aa_vals
    - pca_vals
)

delta_mean, delta_lo, delta_hi = (
    bootstrap_ci(
        delta,
        seed=(
            FIGURE_RNG_SEED
            + 800
        ),
    )
)

axB.text(
    0.5,
    0.97,
    (
        f"Mean paired Δr = "
        f"{delta_mean:.3f}\n"
        f"95% CI "
        f"[{delta_lo:.3f}, "
        f"{delta_hi:.3f}]"
    ),
    transform=axB.transAxes,
    ha="center",
    va="top",
    fontsize=9,
)

axB.set_xticks(
    [
        0,
        1,
    ]
)

axB.set_xticklabels(
    [
        "PCA",
        "MS-AA",
    ]
)

axB.set_xlim(
    -0.35,
    1.35,
)

axB.set_ylim(
    0.50,
    0.90,
)

axB.set_ylabel(
    "Spatial configuration recovery (r)"
)

axB.set_title(
    "B  Metastable simulations",
    loc="left",
    fontweight="bold",
)

axB.spines[
    "top"
].set_visible(
    False
)

axB.spines[
    "right"
].set_visible(
    False
)


# ============================================================
# PANEL C
# Direct recovery versus span recovery
# ============================================================

metric_specs = [
    (
        "Direct configuration",
        "aa_motif_recovery",
        "pca_motif_recovery",
    ),
    (
        "Full latent span",
        "aa_span_recovery",
        "pca_span_recovery",
    ),
]

x_metric = np.arange(
    len(
        metric_specs
    )
)

for method_i, method in enumerate([
    "MS-AA",
    "PCA",
]):

    means = []
    los = []
    his = []

    for metric_i, (
        metric_label,
        aa_col,
        pca_col,
    ) in enumerate(
        metric_specs
    ):

        col = (
            aa_col
            if method
            == "MS-AA"
            else pca_col
        )

        vals = (
            meta[
                col
            ]
            .dropna()
            .to_numpy()
        )

        mean, lo, hi = (
            bootstrap_ci(
                vals,
                seed=(
                    FIGURE_RNG_SEED
                    + 1000
                    + 10
                    * metric_i
                    + method_i
                ),
            )
        )

        means.append(
            mean
        )

        los.append(
            lo
        )

        his.append(
            hi
        )

    means = np.asarray(
        means
    )

    los = np.asarray(
        los
    )

    his = np.asarray(
        his
    )

    axC.plot(
        x_metric,
        means,
        marker="o",
        markersize=7,
        linewidth=1.8,
        label=method,
        color=default_colors[
            method_i
        ],
    )

    axC.errorbar(
        x_metric,
        means,
        yerr=np.vstack([
            means
            - los,
            his
            - means,
        ]),
        fmt="none",
        capsize=4,
        linewidth=1.5,
        color=default_colors[
            method_i
        ],
    )


axC.set_xticks(
    x_metric
)

axC.set_xticklabels(
    [
        "Direct\nconfiguration",
        "Full latent\nspan",
    ]
)

axC.set_ylim(
    0.60,
    1.02,
)

axC.set_ylabel(
    "Recovery (r)"
)

axC.set_title(
    "C  Pattern identity vs. span",
    loc="left",
    fontweight="bold",
)

axC.legend(
    frameon=False
)

axC.spines[
    "top"
].set_visible(
    False
)

axC.spines[
    "right"
].set_visible(
    False
)


# ============================================================
# PANEL D
# Representative matched heatmaps
# ============================================================

heat_true = (
    row_standardize_for_plot(
        representative[
            "true"
        ]
    )
)

heat_aa = (
    row_standardize_for_plot(
        representative[
            "aa"
        ]
    )
)

heat_pca = (
    row_standardize_for_plot(
        representative[
            "pca"
        ]
    )
)

all_heat = np.concatenate(
    [
        heat_true.ravel(),
        heat_aa.ravel(),
        heat_pca.ravel(),
    ]
)

lim = np.nanquantile(
    np.abs(
        all_heat
    ),
    0.99,
)

heat_titles = [
    "True configurations",
    "Matched MS-AA archetypes",
    "Matched PCA components",
]

heat_data = [
    heat_true,
    heat_aa,
    heat_pca,
]

last_im = None

for ax, data, title in zip(
    heat_axes,
    heat_data,
    heat_titles,
):

    last_im = ax.imshow(
        data,
        aspect="auto",
        interpolation="nearest",
        cmap="RdBu_r",
        vmin=-lim,
        vmax=lim,
    )

    add_network_boundaries(
        ax
    )

    ax.set_title(
        title,
        fontsize=10,
    )

    ax.set_xlabel(
        "Nodes ordered by simulated network"
    )

    ax.set_yticks(
        np.arange(
            N_TRUE_MOTIFS
        )
    )

    ax.set_yticklabels(
        [
            str(
                k + 1
            )
            for k in range(
                N_TRUE_MOTIFS
            )
        ]
    )

heat_axes[
    0
].set_ylabel(
    "Latent configuration"
)

for ax in heat_axes[
    1:
]:
    ax.set_yticklabels(
        []
    )

heat_axes[
    0
].text(
    -0.18,
    1.13,
    "D",
    transform=heat_axes[
        0
    ].transAxes,
    fontsize=12,
    fontweight="bold",
    va="top",
)

heat_axes[
    0
].text(
    -0.10,
    1.13,
    "Representative matched spatial patterns",
    transform=heat_axes[
        0
    ].transAxes,
    fontsize=11,
    fontweight="bold",
    va="top",
)

cbar = fig.colorbar(
    last_im,
    ax=heat_axes,
    fraction=0.018,
    pad=0.015,
)

cbar.set_label(
    "Row-standardized pattern value"
)


# ============================================================
# Final formatting + save
# ============================================================

fig.suptitle(
    (
        "Simulation benchmark distinguishes direct "
        "configuration recovery from low-dimensional span preservation"
    ),
    y=0.99,
    fontsize=14,
)

main_png = (
    OUTPUT_DIR
    / "main_simulation_figure.png"
)

main_pdf = (
    OUTPUT_DIR
    / "main_simulation_figure.pdf"
)

fig.savefig(
    main_png,
    dpi=400,
    bbox_inches="tight",
)

fig.savefig(
    main_pdf,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    main_png,
)

print(
    "Saved:",
    main_pdf,
)

In [ ]:
# ============================================================
# SUPPLEMENTARY FIGURE:
# OVERLAP AND DYNAMICS SENSITIVITY
# ============================================================

results_df = load_results()

overlap_df = results_df[
    results_df[
        "regime"
    ]
    == "metastable_overlap"
].copy()

dynamics_df = results_df[
    results_df[
        "regime"
    ]
    == "metastable_dynamics"
].copy()


fig, axes = plt.subplots(
    2,
    2,
    figsize=(
        10,
        8,
    ),
)

default_colors = (
    plt.rcParams[
        "axes.prop_cycle"
    ]
    .by_key()[
        "color"
    ]
)


# ============================================================
# Helper for sensitivity panels
# ============================================================

def plot_sensitivity(
    ax,
    df,
    order,
    labels,
    aa_col,
    pca_col,
    title,
    ylabel,
    seed_offset,
):

    x = np.arange(
        len(order)
    )

    for method_i, (
        method,
        col,
    ) in enumerate([
        (
            "MS-AA",
            aa_col,
        ),
        (
            "PCA",
            pca_col,
        ),
    ]):

        means = []
        los = []
        his = []

        for condition_i, condition in enumerate(
            order
        ):

            vals = (
                df.loc[
                    df[
                        "condition"
                    ]
                    == condition,
                    col,
                ]
                .dropna()
                .to_numpy()
            )

            mean, lo, hi = (
                bootstrap_ci(
                    vals,
                    seed=(
                        FIGURE_RNG_SEED
                        + seed_offset
                        + 100
                        * condition_i
                        + method_i
                    ),
                )
            )

            means.append(
                mean
            )

            los.append(
                lo
            )

            his.append(
                hi
            )

        means = np.asarray(
            means
        )

        los = np.asarray(
            los
        )

        his = np.asarray(
            his
        )

        ax.plot(
            x,
            means,
            marker="o",
            linewidth=1.8,
            label=method,
            color=default_colors[
                method_i
            ],
        )

        ax.errorbar(
            x,
            means,
            yerr=np.vstack([
                means
                - los,
                his
                - means,
            ]),
            fmt="none",
            capsize=4,
            color=default_colors[
                method_i
            ],
        )

    ax.set_xticks(
        x
    )

    ax.set_xticklabels(
        labels
    )

    ax.set_ylim(
        0.45,
        0.90,
    )

    ax.set_ylabel(
        ylabel
    )

    ax.set_title(
        title,
        loc="left",
        fontweight="bold",
    )

    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )


# ============================================================
# Overlap
# ============================================================

overlap_order = [
    "low_overlap",
    "baseline_overlap",
    "high_overlap",
]

overlap_labels = [
    "Low",
    "Baseline",
    "High",
]

plot_sensitivity(
    axes[
        0,
        0,
    ],
    overlap_df,
    overlap_order,
    overlap_labels,
    "aa_motif_recovery",
    "pca_motif_recovery",
    "A  Spatial recovery vs. motif overlap",
    "Spatial recovery (r)",
    10000,
)

plot_sensitivity(
    axes[
        1,
        0,
    ],
    overlap_df,
    overlap_order,
    overlap_labels,
    "aa_temporal_recovery",
    "pca_temporal_recovery",
    "C  Temporal recovery vs. motif overlap",
    "Temporal recovery (r)",
    11000,
)


# ============================================================
# Dynamics
# ============================================================

dynamics_order = [
    "state_like",
    "baseline",
    "more_mixed",
]

dynamics_labels = [
    "State-like",
    "Baseline",
    "More mixed",
]

plot_sensitivity(
    axes[
        0,
        1,
    ],
    dynamics_df,
    dynamics_order,
    dynamics_labels,
    "aa_motif_recovery",
    "pca_motif_recovery",
    "B  Spatial recovery vs. dynamical mixing",
    "Spatial recovery (r)",
    12000,
)

plot_sensitivity(
    axes[
        1,
        1,
    ],
    dynamics_df,
    dynamics_order,
    dynamics_labels,
    "aa_temporal_recovery",
    "pca_temporal_recovery",
    "D  Temporal recovery vs. dynamical mixing",
    "Temporal recovery (r)",
    13000,
)


axes[
    0,
    0
].legend(
    frameon=False
)

fig.tight_layout()

supp_png = (
    OUTPUT_DIR
    / "supp_overlap_dynamics.png"
)

supp_pdf = (
    OUTPUT_DIR
    / "supp_overlap_dynamics.pdf"
)

fig.savefig(
    supp_png,
    dpi=400,
    bbox_inches="tight",
)

fig.savefig(
    supp_pdf,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ============================================================
# CAPTION-READY MAIN FIGURE STATISTICS
# ============================================================

results_df = load_results()

meta = (
    results_df[
        results_df[
            "regime"
        ]
        == "metastable"
    ]
    .copy()
)

aa_spatial = meta[
    "aa_motif_recovery"
].to_numpy()

pca_spatial = meta[
    "pca_motif_recovery"
].to_numpy()

aa_span = meta[
    "aa_span_recovery"
].to_numpy()

pca_span = meta[
    "pca_span_recovery"
].to_numpy()

delta = (
    aa_spatial
    - pca_spatial
)


aa_mean, aa_lo, aa_hi = (
    bootstrap_ci(
        aa_spatial,
        seed=20001,
    )
)

pca_mean, pca_lo, pca_hi = (
    bootstrap_ci(
        pca_spatial,
        seed=20002,
    )
)

delta_mean, delta_lo, delta_hi = (
    bootstrap_ci(
        delta,
        seed=20003,
    )
)

aa_span_mean, aa_span_lo, aa_span_hi = (
    bootstrap_ci(
        aa_span,
        seed=20004,
    )
)

pca_span_mean, pca_span_lo, pca_span_hi = (
    bootstrap_ci(
        pca_span,
        seed=20005,
    )
)


print(
    "\nMETASTABLE SPATIAL RECOVERY"
)

print(
    f"MS-AA: {aa_mean:.3f} "
    f"[{aa_lo:.3f}, {aa_hi:.3f}]"
)

print(
    f"PCA:   {pca_mean:.3f} "
    f"[{pca_lo:.3f}, {pca_hi:.3f}]"
)

print(
    "\nPAIRED DIFFERENCE"
)

print(
    f"AA - PCA: {delta_mean:.3f} "
    f"[{delta_lo:.3f}, {delta_hi:.3f}]"
)

print(
    "AA > PCA in",
    np.sum(
        delta > 0
    ),
    "of",
    len(
        delta
    ),
    "replicates",
)

print(
    "\nSPAN RECOVERY"
)

print(
    f"MS-AA: {aa_span_mean:.3f} "
    f"[{aa_span_lo:.3f}, {aa_span_hi:.3f}]"
)

print(
    f"PCA:   {pca_span_mean:.3f} "
    f"[{pca_span_lo:.3f}, {pca_span_hi:.3f}]"
)

print(
    "\nRepresentative heatmap seed:",
    representative[
        "seed"
    ],
)